[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_3_5/12_tag_3_5_batchsize_epochen.ipynb)

# Tag 3-5 - 12 Batch Size, Epochen und Overfitting

Dieses Notebook zeigt Batch Size und Epochen auf einem lernbaren, aber nicht trivialen Datensatz. Damit Overfitting sichtbar wird, werden nur im Trainingsset einige Labels verrauscht. Validierung und Test bleiben sauber.

Die Test Accuracy wird pro Epoche gemessen. Der Slider zeigt also echte Werte der jeweiligen Epoche.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import ipywidgets as widgets
from IPython.display import display
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("Setup abgeschlossen.")


In [ ]:
X, y_clean = make_classification(
    n_samples=1600,
    n_features=50,
    n_informative=14,
    n_redundant=10,
    class_sep=1.15,
    flip_y=0.02,
    random_state=RANDOM_STATE,
)
X = StandardScaler().fit_transform(X).astype("float32")
y_clean = y_clean.astype("float32")

X_train, X_rest, y_train_clean, y_rest = train_test_split(
    X, y_clean, train_size=420, random_state=RANDOM_STATE, stratify=y_clean
)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=RANDOM_STATE, stratify=y_rest
)

rng = np.random.default_rng(RANDOM_STATE)
y_train_noisy = y_train_clean.copy()
noisy_idx = rng.choice(len(y_train_noisy), size=int(0.18 * len(y_train_noisy)), replace=False)
y_train_noisy[noisy_idx] = 1.0 - y_train_noisy[noisy_idx]

y_train_noisy = y_train_noisy.reshape(-1, 1)
y_val = y_val.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)
print("Verrauschte Trainingslabels:", len(noisy_idx), "von", len(y_train_noisy))


def build_model():
    model = keras.Sequential(
        [
            keras.Input(shape=(X_train.shape[1],), name="features"),
            layers.Dense(128, activation="relu", name="hidden_1"),
            layers.Dense(96, activation="relu", name="hidden_2"),
            layers.Dense(48, activation="relu", name="hidden_3"),
            layers.Dense(1, activation="sigmoid", name="output"),
        ]
    )
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0008), loss="binary_crossentropy", metrics=["accuracy"])
    return model


class TestAccuracyHistory(callbacks.Callback):
    def __init__(self, X_test, y_test):
        super().__init__()
        self.X_test = X_test
        self.y_test = y_test
        self.values = []

    def on_epoch_end(self, epoch, logs=None):
        _, accuracy = self.model.evaluate(self.X_test, self.y_test, verbose=0)
        self.values.append(accuracy)


## Batch Size vergleichen

Kleine Batches führen zu mehr Updates pro Epoche und stärker schwankenden Gradienten. Große Batches sind stabiler, liefern aber weniger Updates pro Epoche.


In [ ]:
batch_sizes = [8, 32, 128]
histories = {}
test_accuracy_by_epoch = {}

for batch_size in batch_sizes:
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = build_model()
    test_callback = TestAccuracyHistory(X_test, y_test)
    history = model.fit(
        X_train,
        y_train_noisy,
        validation_data=(X_val, y_val),
        epochs=80,
        batch_size=batch_size,
        verbose=0,
        callbacks=[test_callback],
    )
    updates_per_epoch = int(np.ceil(len(X_train) / batch_size))
    histories[batch_size] = history
    test_accuracy_by_epoch[batch_size] = test_callback.values
    best_test = max(test_callback.values)
    final_test = test_callback.values[-1]
    print(f"Batch Size {batch_size:>3}: Updates/Epoche={updates_per_epoch:>3}, beste/finale Test Accuracy={best_test:.1%}/{final_test:.1%}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for batch_size, history in histories.items():
    axes[0].plot(history.history["val_loss"], label=f"batch={batch_size}")
    axes[1].plot(history.history["val_accuracy"], label=f"batch={batch_size}")
    axes[2].plot(test_accuracy_by_epoch[batch_size], label=f"batch={batch_size}")
axes[0].set_title("Validation Loss")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoche")
axes[1].legend()
axes[2].set_title("Test Accuracy pro Epoche")
axes[2].set_xlabel("Epoche")
axes[2].legend()
plt.tight_layout()
plt.show()


## Slider: Zustand nach einer bestimmten Epoche

Der Slider zeigt Training, Validation und Test für genau diese Epoche.


In [ ]:
def show_epoch(epoch=10):
    rows = []
    for batch_size, history in histories.items():
        idx = min(epoch - 1, len(history.history["loss"]) - 1)
        rows.append(
            {
                "batch_size": batch_size,
                "epoch": idx + 1,
                "train_loss_noisy": history.history["loss"][idx],
                "val_loss_clean": history.history["val_loss"][idx],
                "train_accuracy_noisy": history.history["accuracy"][idx],
                "val_accuracy_clean": history.history["val_accuracy"][idx],
                "test_accuracy_clean": test_accuracy_by_epoch[batch_size][idx],
            }
        )
    display(pd.DataFrame(rows).round(4))


widgets.interact(
    show_epoch,
    epoch=widgets.IntSlider(value=10, min=1, max=80, step=1, description="Epoche"),
);


## Voll trainieren vs. Early Stopping

Beim Volltraining kann das Modell länger versuchen, die verrauschten Trainingslabels zu erklären. Early Stopping beendet Training anhand sauberer Validierungsdaten.


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
full_model = build_model()
full_test_callback = TestAccuracyHistory(X_test, y_test)
full_history = full_model.fit(
    X_train,
    y_train_noisy,
    validation_data=(X_val, y_val),
    epochs=140,
    batch_size=32,
    verbose=0,
    callbacks=[full_test_callback],
)
full_test_loss, full_test_accuracy = full_model.evaluate(X_test, y_test, verbose=0)

tf.keras.utils.set_random_seed(RANDOM_STATE)
early_model = build_model()
early_test_callback = TestAccuracyHistory(X_test, y_test)
early = callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True)
early_history = early_model.fit(
    X_train,
    y_train_noisy,
    validation_data=(X_val, y_val),
    epochs=140,
    batch_size=32,
    verbose=0,
    callbacks=[early_test_callback, early],
)
early_test_loss, early_test_accuracy = early_model.evaluate(X_test, y_test, verbose=0)

best_epoch_full = int(np.argmin(full_history.history["val_loss"]) + 1)
best_epoch_early = int(np.argmin(early_history.history["val_loss"]) + 1)

print(f"Volltraining: Epochen=140, beste Val-Epoche={best_epoch_full}, finale Test Accuracy={full_test_accuracy:.1%}, beste Test Accuracy={max(full_test_callback.values):.1%}")
print(f"Early Stopping: Epochen={len(early_history.history['loss'])}, beste Val-Epoche={best_epoch_early}, Test Accuracy nach Restore={early_test_accuracy:.1%}, beste gemessene Test Accuracy={max(early_test_callback.values):.1%}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(full_history.history["loss"], label="full train noisy")
axes[0].plot(full_history.history["val_loss"], label="full validation clean")
axes[0].plot(early_history.history["val_loss"], label="early validation clean")
axes[0].axvline(best_epoch_full - 1, color="tab:blue", linestyle="--", alpha=0.7, label="beste full Val-Epoche")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoche")
axes[0].legend(fontsize=8)

axes[1].plot(full_history.history["accuracy"], label="full train noisy")
axes[1].plot(full_history.history["val_accuracy"], label="full validation clean")
axes[1].plot(early_history.history["val_accuracy"], label="early validation clean")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoche")
axes[1].legend(fontsize=8)

axes[2].plot(full_test_callback.values, label="full test clean")
axes[2].plot(early_test_callback.values, label="early test clean")
axes[2].set_title("Test Accuracy pro Epoche")
axes[2].set_xlabel("Epoche")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
plt.bar(["voll trainiert", "early stopping"], [full_test_accuracy, early_test_accuracy])
plt.ylim(0.0, 1.0)
plt.ylabel("Test Accuracy")
plt.title("Finale Test Accuracy")
plt.show()
